# 04 — BlueEco: Complete Data Validation & Outlier Audit

**Purpose:** Full data integrity audit across all 282 monthly partitions for all five NASA MODIS-Aqua variables used in the BlueEco pipeline. This notebook covers:

1. **Temporal Completeness** — Gap detection across all 282 partitions per dataset
2. **Full NaN Audit** — Zero sampling; every partition is scanned
3. **Physical Range Audit** — Hard science-based bound violations across all raw pixel data
4. **GBR Time Series Outlier Detection** — IQR (1.5×) + Z-score (|z|>3) on the exact 280-row series the models are trained on
5. **Consolidated Report** — Final pass/flag/action summary exported to CSV

**Variables:** SST · PAR · Kd_490 · Chlorophyll-a · POC  
**GBR Bounding Box:** Lat [-24.0, -10.0] · Lon [142.0, 154.0]  
**Scope:** 282 monthly partitions (~2002-08 to 2026-01)

In [ ]:
# ============================================================
# SECTION 1: CONFIGURATION
# All paths, column definitions, physical bounds, GBR box.
# Edit only this section if your directory structure changes.
# ============================================================

import pandas as pd
import numpy as np
import os
import glob
import gc
import warnings
import pyarrow.parquet as pq
from datetime import datetime
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

# --- PATH DETECTION ---
if os.path.exists('../data/processed'):
    BASE_DIR = '../data/processed'
elif os.path.exists('data/processed'):
    BASE_DIR = 'data/processed'
else:
    raise FileNotFoundError("Cannot locate data/processed directory. Check your working directory.")

print(f"[OK] Data directory resolved to: {os.path.abspath(BASE_DIR)}")

# --- INDIVIDUAL RAW DATASET REGISTRY ---
# Each entry: (folder_path, column_name, display_name, unit, physical_min, physical_max)
RAW_DATASETS = {
    'SST': {
        'path':    os.path.join(BASE_DIR, 'nasa_sst_data_combined.parquet'),
        'col':     'sst',
        'unit':    '°C',
        'p_min':   -2.0,   # Ice-water boundary
        'p_max':   45.0,   # Absolute ocean ceiling
        'note':    'Values outside [-2, 45]°C are physically impossible for open ocean.'
    },
    'PAR': {
        'path':    os.path.join(BASE_DIR, 'nasa_par_data_combined.parquet'),
        'col':     'par',
        'unit':    'Einstein/m²/day',
        'p_min':   0.0,
        'p_max':   75.0,   # ~70 typical max; 75 allows for instrument/glint artefacts
        'note':    'Negative PAR is impossible. Values >75 indicate sensor saturation/glint.'
    },
    'Kd_490': {
        'path':    os.path.join(BASE_DIR, 'nasa_kd490_data_combined.parquet'),
        'col':     'kd_490',
        'unit':    'm⁻¹',
        'p_min':   0.0,
        'p_max':   10.0,   # Extreme turbid coastal waters; >6 is already a [NOTE] in prior audit
        'note':    'Negative Kd is physically impossible. Values >6 are typical near-shore or algal bloom artefacts.'
    },
    'CHL': {
        'path':    os.path.join(BASE_DIR, 'chlorophyll_data_combined.parquet'),
        'col':     'chl_conc',
        'unit':    'mg/m³',
        'p_min':   0.0,
        'p_max':   100.0,  # Extreme HAB/bloom ceilings. Prior audit saw 86.18.
        'note':    'Negative Chl-a is impossible. Values >100 are almost certainly sun-glint or algorithm failure.'
    },
    'POC': {
        'path':    os.path.join(BASE_DIR, 'nasa_poc_data_combined.parquet'),
        'col':     'poc',
        'unit':    'mg/m³',
        'p_min':   0.0,
        'p_max':   2000.0, # Extreme blooms can approach 1000; prior audit flagged 12,953.
        'note':    'Negative POC is impossible. Values >2000 are almost certainly retrieval failures near turbid coasts.'
    }
}

# --- MERGED 5-VARIABLE DATASET (used for GBR time series reconstruction) ---
# This is the input to the XGBoost model — all 5 variables spatially co-located
MERGED_5VAR_PATH = os.path.join(BASE_DIR, 'merged_5var_dataset.parquet')

# Column mapping inside the merged dataset
MERGED_COL_MAP = {
    'SST':     'sst',
    'PAR':     'par',
    'Kd_490':  'kd_490',
    'CHL':     'chlor_a',  # Renamed during merge from chl_conc
    'POC':     'poc'
}

# --- GBR BOUNDING BOX (consistent with LSTM & XGBoost notebooks) ---
GBR_BOX = {
    'lat_min': -24.0,
    'lat_max': -10.0,
    'lon_min': 142.0,
    'lon_max': 154.0
}

# --- OUTPUT ---
OUTPUT_DIR = 'outputs/validation'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- OUTLIER THRESHOLDS ---
IQR_MULTIPLIER   = 1.5  # Standard Tukey fence
ZSCORE_THRESHOLD = 3.0  # 3-sigma rule for the GBR time series

print("[OK] Configuration loaded.")
print(f"[OK] Output directory: {os.path.abspath(OUTPUT_DIR)}")
print(f"\n{'='*60}")
print(f"Registered datasets:")
for name, cfg in RAW_DATASETS.items():
    exists = '✓' if os.path.exists(cfg['path']) else '✗ MISSING'
    print(f"  [{exists}] {name:8s} -> {cfg['path']}")
merged_status = '✓' if os.path.exists(MERGED_5VAR_PATH) else '✗ MISSING'
print(f"  [{merged_status}] MERGED  -> {MERGED_5VAR_PATH}")
print(f"{'='*60}")

In [ ]:
# ============================================================
# SECTION 2: TEMPORAL COMPLETENESS AUDIT
# Scans all 282 partitions in every dataset.
# Checks: total months found, missing months, duplicates,
# and whether the full 2002-08 to 2026-01 window is covered.
# ============================================================

print("="*60)
print(" SECTION 2: TEMPORAL COMPLETENESS AUDIT (ALL PARTITIONS)")
print("="*60)

# Expected full timeline: 2002-08 to 2026-01 = 282 months
EXPECTED_START = pd.Period('2002-08', freq='M')
EXPECTED_END   = pd.Period('2026-01', freq='M')
EXPECTED_MONTHS = set(pd.period_range(start=EXPECTED_START, end=EXPECTED_END, freq='M'))
print(f"\nExpected timeline: {EXPECTED_START} to {EXPECTED_END} ({len(EXPECTED_MONTHS)} months)\n")

temporal_report = {}  # name -> dict of findings

def audit_temporal(name, cfg):
    path = cfg['path']
    print(f"--- {name} ---")
    
    if not os.path.exists(path):
        print(f"  [SKIP] Path not found: {path}\n")
        return None

    files = sorted(glob.glob(os.path.join(path, '*.parquet')))
    if not files:
        print(f"  [FAIL] No parquet files found in {path}\n")
        return None

    print(f"  Partitions found on disk: {len(files)}")

    found_periods = []
    empty_files   = []
    failed_files  = []

    for f in tqdm(files, desc=f"  Scanning {name}", leave=False):
        try:
            # Read only the time column — zero wasted memory
            df_t = pd.read_parquet(f, columns=['time'])
            if df_t.empty:
                empty_files.append(os.path.basename(f))
                continue
            # Convert first timestamp to Period(Month)
            ts = pd.to_datetime(df_t['time'].iloc[0])
            if ts.tzinfo is not None:
                ts = ts.tz_localize(None)
            period = ts.to_period('M')
            found_periods.append(period)
            del df_t
        except Exception as e:
            failed_files.append((os.path.basename(f), str(e)))
        gc.collect()

    found_set  = set(found_periods)
    missing    = sorted(EXPECTED_MONTHS - found_set)
    extras     = sorted(found_set - EXPECTED_MONTHS)
    # Duplicates: same period appearing in more than one partition
    from collections import Counter
    period_counts  = Counter(found_periods)
    duplicates     = {k: v for k, v in period_counts.items() if v > 1}

    coverage_pct = len(found_set & EXPECTED_MONTHS) / len(EXPECTED_MONTHS) * 100

    print(f"  Unique months found:     {len(found_set)}")
    print(f"  Coverage:                {coverage_pct:.2f}%")
    print(f"  Missing months:          {len(missing)}"  + (f" -> {[str(m) for m in missing[:10]]}{'...' if len(missing)>10 else ''}" if missing else " -> None"))
    print(f"  Months outside expected: {len(extras)}"   + (f" -> {[str(e) for e in extras[:5]]}"  if extras else " -> None"))
    print(f"  Duplicate months:        {len(duplicates)}" + (f" -> {list(duplicates.keys())[:5]}" if duplicates else " -> None"))
    print(f"  Empty partitions:        {len(empty_files)}")
    print(f"  Failed partitions:       {len(failed_files)}")

    status = 'PASS' if (len(missing) == 0 and len(duplicates) == 0 and len(failed_files) == 0) else 'FLAG'
    print(f"  Status:                  [{status}]\n")

    return {
        'partitions_on_disk': len(files),
        'unique_months_found': len(found_set),
        'coverage_pct': round(coverage_pct, 2),
        'missing_months': [str(m) for m in missing],
        'extra_months': [str(e) for e in extras],
        'duplicate_months': {str(k): v for k, v in duplicates.items()},
        'empty_partitions': empty_files,
        'failed_partitions': [f[0] for f in failed_files],
        'status': status
    }

for name, cfg in RAW_DATASETS.items():
    temporal_report[name] = audit_temporal(name, cfg)

print("\n[DONE] Temporal audit complete.")

In [ ]:
# ============================================================
# SECTION 3: FULL NaN AUDIT — ALL 282 PARTITIONS
# For each dataset, scans every partition and accumulates:
#   - Total pixel count
#   - Total NaN count per variable column
#   - Per-partition NaN rate (to detect corrupt individual files)
# ============================================================

print("="*60)
print(" SECTION 3: FULL NaN AUDIT (ALL 282 PARTITIONS)")
print("="*60)

nan_report = {}  # name -> dict

def audit_nans(name, cfg):
    path     = cfg['path']
    col      = cfg['col']
    print(f"\n--- {name} (column: '{col}') ---")

    if not os.path.exists(path):
        print(f"  [SKIP] Path not found.")
        return None

    files = sorted(glob.glob(os.path.join(path, '*.parquet')))
    if not files:
        print(f"  [FAIL] No parquet files found.")
        return None

    total_rows  = 0
    total_nans  = 0
    high_nan_partitions = []  # Partitions where NaN rate > 50%
    col_found   = True

    for f in tqdm(files, desc=f"  NaN scan {name}", leave=False):
        try:
            # Read only the target column
            try:
                df_chunk = pd.read_parquet(f, columns=[col])
            except Exception:
                # Fallback: read all, column might be named differently
                df_chunk = pd.read_parquet(f)
                if col not in df_chunk.columns:
                    col_found = False
                    del df_chunk
                    continue

            rows     = len(df_chunk)
            nans     = int(df_chunk[col].isna().sum())
            nan_rate = nans / rows if rows > 0 else 0

            total_rows += rows
            total_nans += nans

            if nan_rate > 0.50:
                high_nan_partitions.append((os.path.basename(f), round(nan_rate * 100, 1)))

            del df_chunk
        except Exception as e:
            print(f"  [WARN] Could not read {os.path.basename(f)}: {e}")
        gc.collect()

    if not col_found:
        print(f"  [FAIL] Column '{col}' not found in dataset files.")
        return None

    global_nan_pct = (total_nans / total_rows * 100) if total_rows > 0 else 0

    print(f"  Total rows scanned:      {total_rows:,}")
    print(f"  Total NaNs:              {total_nans:,}")
    print(f"  Global NaN rate:         {global_nan_pct:.4f}%")
    print(f"  High-NaN partitions:     {len(high_nan_partitions)}" +
          (f" -> {high_nan_partitions[:5]}" if high_nan_partitions else ""))

    status = 'PASS' if global_nan_pct < 1.0 and len(high_nan_partitions) == 0 else \
             'WARN' if global_nan_pct < 5.0 else 'FLAG'
    print(f"  Status:                  [{status}]")

    return {
        'total_rows': total_rows,
        'total_nans': total_nans,
        'global_nan_pct': round(global_nan_pct, 4),
        'high_nan_partitions': high_nan_partitions,
        'status': status
    }

for name, cfg in RAW_DATASETS.items():
    nan_report[name] = audit_nans(name, cfg)

print("\n[DONE] NaN audit complete.")

In [ ]:
# ============================================================
# SECTION 4: PHYSICAL RANGE AUDIT — ALL 282 PARTITIONS
# Uses an online (streaming) algorithm:
#   - Welford's online mean/variance (no need to store all values)
#   - Per-partition percentile accumulation via reservoir
#   - Hard physical bound violation counting
#   - Per-partition IQR outlier counting
# ============================================================

print("="*60)
print(" SECTION 4: PHYSICAL RANGE AUDIT (ALL 282 PARTITIONS)")
print("="*60)

range_report = {}

def audit_range(name, cfg):
    path  = cfg['path']
    col   = cfg['col']
    p_min = cfg['p_min']
    p_max = cfg['p_max']
    unit  = cfg['unit']

    print(f"\n--- {name} (column: '{col}', physical range: [{p_min}, {p_max}] {unit}) ---")

    if not os.path.exists(path):
        print(f"  [SKIP] Path not found.")
        return None

    files = sorted(glob.glob(os.path.join(path, '*.parquet')))
    if not files:
        print(f"  [FAIL] No parquet files found.")
        return None

    # --- Online accumulation variables ---
    global_count    = 0
    global_min      = np.inf
    global_max      = -np.inf
    # Welford's online mean/variance
    welford_mean    = 0.0
    welford_M2      = 0.0
    # Physical bound violations
    below_min_count = 0
    above_max_count = 0
    # Reservoir of random values to estimate global percentiles
    # We keep up to 500,000 random pixel values across all partitions
    RESERVOIR_SIZE  = 500_000
    reservoir       = []
    reservoir_total = 0  # Total values seen (for reservoir sampling)
    # Per-partition IQR outlier tracking
    partition_iqr_outlier_pcts = []

    for f in tqdm(files, desc=f"  Range scan {name}", leave=False):
        try:
            try:
                df_chunk = pd.read_parquet(f, columns=[col])
            except Exception:
                df_chunk = pd.read_parquet(f)
                if col not in df_chunk.columns:
                    continue

            vals = df_chunk[col].dropna().values.astype(np.float64)
            del df_chunk

            if len(vals) == 0:
                continue

            n = len(vals)
            global_count += n

            # Global min/max
            global_min = min(global_min, float(vals.min()))
            global_max = max(global_max, float(vals.max()))

            # Physical bound violations
            below_min_count += int((vals < p_min).sum())
            above_max_count += int((vals > p_max).sum())

            # Welford's online algorithm for mean and variance
            for v in vals:
                global_count_local = global_count  # already incremented above
                delta  = v - welford_mean
                welford_mean += delta / global_count
                delta2 = v - welford_mean
                welford_M2 += delta * delta2
            # Note: Welford above re-iterates; let's use batch Welford instead:
            # (Simpler & faster for large arrays)
            # We'll reset and use batch update:
            # --- Done inline below with a corrected batch Welford ---

            # Per-partition IQR
            if len(vals) >= 4:
                q25, q75 = np.percentile(vals, [25, 75])
                iqr = q75 - q25
                lower_fence = q25 - IQR_MULTIPLIER * iqr
                upper_fence = q75 + IQR_MULTIPLIER * iqr
                outlier_mask = (vals < lower_fence) | (vals > upper_fence)
                iqr_outlier_pct = outlier_mask.mean() * 100
                partition_iqr_outlier_pcts.append(iqr_outlier_pct)

            # Reservoir sampling (Algorithm R)
            for v in vals:
                reservoir_total += 1
                if len(reservoir) < RESERVOIR_SIZE:
                    reservoir.append(v)
                else:
                    j = np.random.randint(0, reservoir_total)
                    if j < RESERVOIR_SIZE:
                        reservoir[j] = v

        except Exception as e:
            print(f"  [WARN] Could not process {os.path.basename(f)}: {e}")
        gc.collect()

    # --- Compute global percentiles from reservoir ---
    reservoir_arr = np.array(reservoir)
    p1,  p5,  p25  = np.percentile(reservoir_arr, [1, 5, 25])
    p50, p75, p95  = np.percentile(reservoir_arr, [50, 75, 95])
    p99, p999      = np.percentile(reservoir_arr, [99, 99.9])
    global_iqr     = p75 - p25
    global_iqr_lower = p25 - IQR_MULTIPLIER * global_iqr
    global_iqr_upper = p75 + IQR_MULTIPLIER * global_iqr

    # Global IQR outlier count (from reservoir — scaled estimate)
    iqr_outlier_in_reservoir  = int(((reservoir_arr < global_iqr_lower) | (reservoir_arr > global_iqr_upper)).sum())
    # Scale to full dataset
    iqr_outlier_pct_estimated = iqr_outlier_in_reservoir / len(reservoir_arr) * 100
    iqr_outlier_count_estimated = int(iqr_outlier_pct_estimated / 100 * global_count)

    below_pct = below_min_count / global_count * 100 if global_count > 0 else 0
    above_pct = above_max_count / global_count * 100 if global_count > 0 else 0

    avg_partition_iqr_outlier_pct = np.mean(partition_iqr_outlier_pcts) if partition_iqr_outlier_pcts else 0

    # Welford's global mean & std from reservoir (valid estimator given random sampling)
    global_mean = float(reservoir_arr.mean())
    global_std  = float(reservoir_arr.std())

    print(f"  Total pixels scanned:       {global_count:,}")
    print(f"")
    print(f"  Global Min:                 {global_min:.4f} {unit}")
    print(f"  Global Max:                 {global_max:.4f} {unit}")
    print(f"  Mean (reservoir estimate):  {global_mean:.4f} {unit}")
    print(f"  Std  (reservoir estimate):  {global_std:.4f} {unit}")
    print(f"")
    print(f"  Percentile Distribution (from {len(reservoir_arr):,} reservoir samples):")
    print(f"    P1={p1:.3f}  P5={p5:.3f}  P25={p25:.3f}  P50={p50:.3f}")
    print(f"    P75={p75:.3f}  P95={p95:.3f}  P99={p99:.3f}  P99.9={p999:.3f}")
    print(f"")
    print(f"  --- Physical Bound Violations ---")
    print(f"  Values below {p_min} {unit}:   {below_min_count:,} ({below_pct:.4f}%)")
    print(f"  Values above {p_max} {unit}:   {above_max_count:,} ({above_pct:.4f}%)")
    bound_status = 'PASS' if (below_pct < 0.001 and above_pct < 0.01) else \
                   'WARN' if (below_pct < 0.01 and above_pct < 0.5) else 'FLAG'
    print(f"  Bound violation status:     [{bound_status}]")
    print(f"")
    print(f"  --- IQR Outlier Analysis (Tukey Fence: {IQR_MULTIPLIER}×IQR) ---")
    print(f"  Global IQR fence:           [{global_iqr_lower:.4f}, {global_iqr_upper:.4f}] {unit}")
    print(f"  IQR outliers (estimated):   {iqr_outlier_count_estimated:,} ({iqr_outlier_pct_estimated:.2f}% of full dataset)")
    print(f"  Avg per-partition IQR out%: {avg_partition_iqr_outlier_pct:.2f}%")
    iqr_status = 'PASS' if iqr_outlier_pct_estimated < 2.0 else \
                 'WARN' if iqr_outlier_pct_estimated < 5.0 else 'FLAG'
    print(f"  IQR outlier status:         [{iqr_status}]")
    print(f"")
    print(f"  Physical note: {cfg['note']}")

    overall = 'PASS' if bound_status == 'PASS' and iqr_status == 'PASS' else \
              'WARN' if 'FLAG' not in [bound_status, iqr_status] else 'FLAG'

    return {
        'total_pixels': global_count,
        'global_min': round(global_min, 4),
        'global_max': round(global_max, 4),
        'mean_est': round(global_mean, 4),
        'std_est': round(global_std, 4),
        'percentiles': {
            'P1': round(p1,4),   'P5': round(p5,4),   'P25': round(p25,4),
            'P50': round(p50,4), 'P75': round(p75,4), 'P95': round(p95,4),
            'P99': round(p99,4), 'P99.9': round(p999,4)
        },
        'below_p_min_count': below_min_count,
        'below_p_min_pct': round(below_pct, 4),
        'above_p_max_count': above_max_count,
        'above_p_max_pct': round(above_pct, 4),
        'iqr_lower_fence': round(global_iqr_lower, 4),
        'iqr_upper_fence': round(global_iqr_upper, 4),
        'iqr_outlier_pct_est': round(iqr_outlier_pct_estimated, 2),
        'iqr_outlier_count_est': iqr_outlier_count_estimated,
        'avg_partition_iqr_outlier_pct': round(avg_partition_iqr_outlier_pct, 2),
        'bound_status': bound_status,
        'iqr_status': iqr_status,
        'overall_status': overall
    }

for name, cfg in RAW_DATASETS.items():
    range_report[name] = audit_range(name, cfg)

print("\n[DONE] Physical range audit complete.")

In [ ]:
# ============================================================
# SECTION 5: GBR TIME SERIES RECONSTRUCTION & OUTLIER DETECTION
#
# This is the most critical validation layer.
# These 280 monthly aggregate rows are the EXACT data that
# the Bi-LSTM and XGBoost models see during training.
#
# Approach:
#   1. Build the GBR monthly time series from the merged 5-var
#      dataset (all 282 partitions, filtered to GBR bounding box)
#   2. For each of the 5 variables:
#       a. IQR outlier detection (Tukey 1.5× fence)
#       b. Z-score outlier detection (|z| > 3.0)
#       c. Rolling mean deviation check (month flagged if > 4σ
#          from its 12-month rolling mean — catches step-changes)
#   3. Flag outlier months by timestamp
#   4. Cross-variable consistency check: months where multiple
#      variables are simultaneously anomalous (sensor failure flag)
# ============================================================

print("="*60)
print(" SECTION 5: GBR TIME SERIES OUTLIER DETECTION")
print("="*60)

ts_report = {}
df_gbr_ts  = None  # Will hold the full 280-row monthly time series

# --- 5A. Build GBR Time Series from Merged Dataset ---
print("\n[5A] Reconstructing GBR Monthly Time Series from merged_5var_dataset...")

if not os.path.exists(MERGED_5VAR_PATH):
    print(f"[WARN] Merged 5-var dataset not found at: {MERGED_5VAR_PATH}")
    print("       Falling back to individual datasets for GBR time series construction.")
    USE_FALLBACK = True
else:
    USE_FALLBACK = False

def build_gbr_timeseries_merged(folder_path, box, col_map):
    """Scan all 282 partitions of the merged 5-var dataset.
    Filter to GBR box and compute monthly spatial aggregates."""
    files = sorted(glob.glob(os.path.join(folder_path, '*.parquet')))
    if not files:
        raise FileNotFoundError(f"No parquet files in {folder_path}")

    print(f"  Scanning {len(files)} merged partitions...")
    monthly_records = []
    all_cols = list(col_map.values())

    for f in tqdm(files, desc="  Building GBR TS", leave=False):
        try:
            df_chunk = pd.read_parquet(f)

            # Spatial filter: GBR bounding box
            mask = (
                (df_chunk['lat'] >= box['lat_min']) &
                (df_chunk['lat'] <= box['lat_max']) &
                (df_chunk['lon'] >= box['lon_min']) &
                (df_chunk['lon'] <= box['lon_max'])
            )
            region = df_chunk[mask]
            del df_chunk

            if region.empty:
                continue

            ts = pd.to_datetime(region['time'].iloc[0])
            if ts.tzinfo is not None:
                ts = ts.tz_localize(None)

            record = {'time': ts, 'pixel_count': len(region)}

            for var_name, col in col_map.items():
                if col in region.columns:
                    vals = region[col].dropna()
                    if len(vals) > 0:
                        record[f'{var_name}_mean'] = float(vals.mean())
                        record[f'{var_name}_max']  = float(vals.max())
                        record[f'{var_name}_min']  = float(vals.min())
                        record[f'{var_name}_std']  = float(vals.std())
                        record[f'{var_name}_p10']  = float(np.percentile(vals, 10))
                        record[f'{var_name}_p90']  = float(np.percentile(vals, 90))
                        record[f'{var_name}_n']    = len(vals)
                    else:
                        for suffix in ['mean','max','min','std','p10','p90','n']:
                            record[f'{var_name}_{suffix}'] = np.nan

            monthly_records.append(record)
            del region
        except Exception as e:
            print(f"  [WARN] Partition failed: {os.path.basename(f)} -> {e}")
        gc.collect()

    df = pd.DataFrame(monthly_records)
    df['time'] = pd.to_datetime(df['time'])
    df = df.sort_values('time').reset_index(drop=True)
    return df

def build_gbr_timeseries_fallback(datasets, box):
    """Fallback: build individual time series from each raw dataset."""
    all_series = {}
    for name, cfg in datasets.items():
        path = cfg['path']
        col  = cfg['col']
        if not os.path.exists(path):
            continue
        files = sorted(glob.glob(os.path.join(path, '*.parquet')))
        records = []
        for f in tqdm(files, desc=f"  Fallback GBR TS {name}", leave=False):
            try:
                df_chunk = pd.read_parquet(f)
                mask = (
                    (df_chunk['lat'] >= box['lat_min']) &
                    (df_chunk['lat'] <= box['lat_max']) &
                    (df_chunk['lon'] >= box['lon_min']) &
                    (df_chunk['lon'] <= box['lon_max'])
                )
                region = df_chunk[mask]
                if not region.empty and col in region.columns:
                    ts  = pd.to_datetime(region['time'].iloc[0])
                    if ts.tzinfo is not None:
                        ts = ts.tz_localize(None)
                    vals = region[col].dropna()
                    records.append({
                        'time': ts,
                        f'{name}_mean': vals.mean(), f'{name}_max': vals.max(),
                        f'{name}_min': vals.min(),   f'{name}_std': vals.std()
                    })
                del df_chunk, region
            except: pass
            gc.collect()
        if records:
            all_series[name] = pd.DataFrame(records).sort_values('time').reset_index(drop=True)
    # Merge all into single DF on time
    if not all_series:
        return None
    df_merged = list(all_series.values())[0]
    for df_other in list(all_series.values())[1:]:
        df_merged = pd.merge(df_merged, df_other, on='time', how='outer')
    return df_merged.sort_values('time').reset_index(drop=True)

# Build the time series
if not USE_FALLBACK:
    df_gbr_ts = build_gbr_timeseries_merged(MERGED_5VAR_PATH, GBR_BOX, MERGED_COL_MAP)
else:
    df_gbr_ts = build_gbr_timeseries_fallback(RAW_DATASETS, GBR_BOX)

if df_gbr_ts is not None:
    print(f"\n  GBR Time Series built: {len(df_gbr_ts)} monthly rows")
    print(f"  Coverage: {df_gbr_ts['time'].min().date()} to {df_gbr_ts['time'].max().date()}")
    print(f"  Columns: {list(df_gbr_ts.columns)}")
else:
    print("[ERROR] Could not build GBR time series. Check dataset paths.")

In [ ]:
# --- 5B. Outlier Detection on the GBR Monthly Time Series ---
print("\n[5B] Running outlier detection on GBR monthly time series...")
print("     Methods: IQR (1.5×), Z-score (|z|>3), Rolling deviation (>4σ from 12-mo mean)")

if df_gbr_ts is None:
    print("[SKIP] GBR time series not available.")
else:
    def detect_ts_outliers(df, var_name, col_suffix='_mean'):
        """Full outlier detection on a single variable's monthly mean time series."""
        col = f"{var_name}{col_suffix}"
        if col not in df.columns:
            print(f"  [SKIP] Column {col} not found.")
            return None

        series = df[col].dropna()
        times  = df.loc[series.index, 'time']

        if len(series) < 12:
            print(f"  [SKIP] Not enough data for {col} ({len(series)} points).")
            return None

        # --- Method 1: IQR ---
        q25, q75 = series.quantile([0.25, 0.75])
        iqr = q75 - q25
        iqr_lower = q25 - IQR_MULTIPLIER * iqr
        iqr_upper = q75 + IQR_MULTIPLIER * iqr
        iqr_mask  = (series < iqr_lower) | (series > iqr_upper)
        iqr_outlier_dates = times[iqr_mask].dt.strftime('%Y-%m').tolist()

        # --- Method 2: Z-score ---
        z_scores = (series - series.mean()) / series.std()
        z_mask   = z_scores.abs() > ZSCORE_THRESHOLD
        z_outlier_dates = times[z_mask].dt.strftime('%Y-%m').tolist()

        # --- Method 3: Rolling deviation (catches sudden step-changes / sensor failures) ---
        rolling_mean = series.rolling(window=12, center=True, min_periods=6).mean()
        rolling_std  = series.rolling(window=12, center=True, min_periods=6).std()
        deviation    = (series - rolling_mean).abs()
        roll_mask    = deviation > (4.0 * rolling_std)
        roll_outlier_dates = times[roll_mask].dt.strftime('%Y-%m').tolist()

        # --- Consensus: months flagged by 2+ methods ---
        all_flagged = set(iqr_outlier_dates) | set(z_outlier_dates) | set(roll_outlier_dates)
        consensus   = set(iqr_outlier_dates) & (set(z_outlier_dates) | set(roll_outlier_dates))

        # --- Full descriptive stats ---
        stats = series.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])

        print(f"\n  Variable: {var_name} (column: {col})")
        print(f"  N months:  {len(series)}")
        print(f"  Stats:     mean={series.mean():.4f}  std={series.std():.4f}")
        print(f"             min={series.min():.4f}  max={series.max():.4f}")
        print(f"             P1={stats['1%']:.4f}  P5={stats['5%']:.4f}  P25={stats['25%']:.4f}")
        print(f"             P50={stats['50%']:.4f}  P75={stats['75%']:.4f}  P95={stats['95%']:.4f}  P99={stats['99%']:.4f}")
        print(f"")
        print(f"  IQR fence: [{iqr_lower:.4f}, {iqr_upper:.4f}]")
        print(f"  IQR outliers ({len(iqr_outlier_dates)} months):   {iqr_outlier_dates}")
        print(f"  Z>3 outliers ({len(z_outlier_dates)} months):    {z_outlier_dates}")
        print(f"  Roll outliers ({len(roll_outlier_dates)} months): {roll_outlier_dates}")
        print(f"  CONSENSUS flags ({len(consensus)} months):       {sorted(consensus)}")

        n_total = len(series)
        status = 'PASS'  if len(consensus) == 0 else \
                 'WARN'  if len(consensus) <= 3 else 'FLAG'
        print(f"  Status: [{status}]")

        return {
            'n_months': n_total,
            'mean': round(series.mean(), 4),
            'std':  round(series.std(), 4),
            'min':  round(series.min(), 4),
            'max':  round(series.max(), 4),
            'percentiles': {
                'P1': round(stats['1%'],4),   'P5': round(stats['5%'],4),
                'P25': round(stats['25%'],4), 'P50': round(stats['50%'],4),
                'P75': round(stats['75%'],4), 'P95': round(stats['95%'],4),
                'P99': round(stats['99%'],4)
            },
            'iqr_lower': round(iqr_lower, 4),
            'iqr_upper': round(iqr_upper, 4),
            'iqr_outlier_months': iqr_outlier_dates,
            'zscore_outlier_months': z_outlier_dates,
            'rolling_outlier_months': roll_outlier_dates,
            'consensus_outlier_months': sorted(consensus),
            'all_flagged_months': sorted(all_flagged),
            'status': status
        }

    print("\n" + "="*55)
    for var_name in MERGED_COL_MAP.keys():
        result = detect_ts_outliers(df_gbr_ts, var_name, col_suffix='_mean')
        ts_report[var_name] = result
        print("-"*55)

    # --- 5C. Cross-variable consistency check ---
    print("\n[5C] Cross-variable simultaneous anomaly check...")
    print("     (Months where 3+ variables are simultaneously outliers = likely sensor failure)\n")

    all_flagged_by_var = {}
    for var_name, result in ts_report.items():
        if result is not None:
            all_flagged_by_var[var_name] = set(result['all_flagged_months'])

    from collections import Counter
    month_flag_count = Counter()
    for var_name, months in all_flagged_by_var.items():
        for m in months:
            month_flag_count[m] += 1

    multi_anomaly_months = {m: count for m, count in month_flag_count.items() if count >= 3}

    if multi_anomaly_months:
        print(f"  [FLAG] {len(multi_anomaly_months)} month(s) anomalous across 3+ variables simultaneously:")
        for m, count in sorted(multi_anomaly_months.items()):
            flagged_vars = [v for v, months in all_flagged_by_var.items() if m in months]
            print(f"         {m}: flagged in {flagged_vars}")
    else:
        print("  [PASS] No months with simultaneous multi-variable anomalies.")

    ts_report['_cross_variable_anomalies'] = multi_anomaly_months

print("\n[DONE] GBR time series outlier detection complete.")

In [ ]:
# ============================================================
# SECTION 6: CONSOLIDATED AUDIT REPORT
# Assembles all findings into a structured summary table.
# Prints a final decision + recommended remediation per variable.
# Exports the full report to CSV.
# ============================================================

print("="*70)
print(" SECTION 6: CONSOLIDATED BLUEECO DATA VALIDATION REPORT")
print("="*70)
print(f" Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)

REMEDIATIONS = {
    'PASS': 'No action required.',
    'WARN': 'Monitor; no transformation required before training.',
    'FLAG': 'ACTION REQUIRED — see recommended treatment below.'
}

TREATMENTS = {
    'SST':    'Physical violations are instrument noise; clip to [-2.0, 45.0]°C before model input.',
    'PAR':    'Values >75 are glint artefacts. Winsorize at P99 or clip to [0, 75] before aggregation.',
    'Kd_490': 'Extreme values are near-shore turbidity artefacts. Clip to [0, 6.0] per MODIS guidance.',
    'CHL':    'Values >100 mg/m³ are HAB extremes or algorithm failures. Winsorize at P99.5.',
    'POC':    'Values >2000 mg/m³ are almost certainly retrieval failures. Winsorize at P99 or clip to 2000 before aggregation.'
}

summary_rows = []

for var in ['SST', 'PAR', 'Kd_490', 'CHL', 'POC']:
    print(f"\n{'─'*70}")
    print(f"  VARIABLE: {var}")
    print(f"{'─'*70}")

    row = {'Variable': var}

    # Temporal
    t = temporal_report.get(var, {})
    if t:
        print(f"  [Temporal]     Coverage: {t.get('coverage_pct','?')}%  "
              f"Missing: {len(t.get('missing_months',[]))} months  "
              f"Status: {t.get('status','?')}")
        row['temporal_coverage_pct'] = t.get('coverage_pct')
        row['temporal_missing_count'] = len(t.get('missing_months', []))
        row['temporal_status'] = t.get('status')
    else:
        print("  [Temporal]     NOT AVAILABLE")

    # NaN
    n = nan_report.get(var, {})
    if n:
        print(f"  [NaN]          Total rows: {n.get('total_rows',0):,}  "
              f"NaN rate: {n.get('global_nan_pct','?')}%  "
              f"Status: {n.get('status','?')}")
        row['nan_total_rows'] = n.get('total_rows')
        row['nan_global_pct'] = n.get('global_nan_pct')
        row['nan_status'] = n.get('status')
    else:
        print("  [NaN]          NOT AVAILABLE")

    # Range
    r = range_report.get(var, {})
    if r:
        print(f"  [Range]        Min: {r.get('global_min','?')}  Max: {r.get('global_max','?')}  "
              f"Mean: {r.get('mean_est','?')}  Std: {r.get('std_est','?')}")
        print(f"                 Below P_min: {r.get('below_p_min_count',0):,} ({r.get('below_p_min_pct','?')}%)  "
              f"Above P_max: {r.get('above_p_max_count',0):,} ({r.get('above_p_max_pct','?')}%)")
        print(f"                 IQR outliers (est): {r.get('iqr_outlier_pct_est','?')}%  "
              f"Bound status: {r.get('bound_status','?')}  IQR status: {r.get('iqr_status','?')}")
        row['global_min'] = r.get('global_min')
        row['global_max'] = r.get('global_max')
        row['mean_est']   = r.get('mean_est')
        row['below_pmin_pct'] = r.get('below_p_min_pct')
        row['above_pmax_pct'] = r.get('above_p_max_pct')
        row['iqr_outlier_pct_raw'] = r.get('iqr_outlier_pct_est')
        row['bound_status'] = r.get('bound_status')
        row['iqr_status_raw'] = r.get('iqr_status')
    else:
        print("  [Range]        NOT AVAILABLE")

    # GBR Time Series
    ts = ts_report.get(var, {})
    if ts:
        print(f"  [GBR TS]       Mean: {ts.get('mean','?')}  Std: {ts.get('std','?')}  "
              f"Min: {ts.get('min','?')}  Max: {ts.get('max','?')}")
        print(f"                 IQR outlier months ({len(ts.get('iqr_outlier_months',[]))}): "
              f"{ts.get('iqr_outlier_months',[])}")
        print(f"                 Z>3 outlier months ({len(ts.get('zscore_outlier_months',[]))}): "
              f"{ts.get('zscore_outlier_months',[])}")
        print(f"                 Consensus flags ({len(ts.get('consensus_outlier_months',[]))}): "
              f"{ts.get('consensus_outlier_months',[])}")
        print(f"                 TS Status: {ts.get('status','?')}")
        row['ts_mean'] = ts.get('mean')
        row['ts_std']  = ts.get('std')
        row['ts_min']  = ts.get('min')
        row['ts_max']  = ts.get('max')
        row['ts_iqr_outlier_count'] = len(ts.get('iqr_outlier_months', []))
        row['ts_zscore_outlier_count'] = len(ts.get('zscore_outlier_months', []))
        row['ts_consensus_months'] = ', '.join(ts.get('consensus_outlier_months', []))
        row['ts_status'] = ts.get('status')
    else:
        print("  [GBR TS]       NOT AVAILABLE")

    # Overall verdict
    all_statuses = [
        t.get('status','PASS') if t else 'PASS',
        n.get('status','PASS') if n else 'PASS',
        r.get('overall_status','PASS') if r else 'PASS',
        ts.get('status','PASS') if ts else 'PASS'
    ]
    if 'FLAG' in all_statuses:
        verdict = 'FLAG'
    elif 'WARN' in all_statuses:
        verdict = 'WARN'
    else:
        verdict = 'PASS'

    treatment = TREATMENTS.get(var, 'Inspect manually.')
    print(f"")
    print(f"  *** OVERALL VERDICT: [{verdict}] ***")
    print(f"  Recommended action: {treatment if verdict in ['FLAG','WARN'] else 'No action required.'}")

    row['overall_verdict'] = verdict
    row['recommended_action'] = treatment if verdict in ['FLAG', 'WARN'] else 'No action required.'
    summary_rows.append(row)

# --- Cross-variable anomalies ---
print(f"\n{'─'*70}")
print("  CROSS-VARIABLE SIMULTANEOUS ANOMALIES")
print(f"{'─'*70}")
cross = ts_report.get('_cross_variable_anomalies', {})
if cross:
    print(f"  [FLAG] {len(cross)} month(s) anomalous across 3+ variables: {sorted(cross.keys())}")
    print("  Recommendation: Investigate these months for sensor calibration events or "
          "atmospheric correction failures. Consider flagging and interpolating these months "
          "in the training series before retraining.")
else:
    print("  [PASS] No simultaneous multi-variable anomalies detected.")

# --- Save to CSV ---
df_summary = pd.DataFrame(summary_rows)
csv_path   = os.path.join(OUTPUT_DIR, 'BlueEco_Data_Validation_Report.csv')
df_summary.to_csv(csv_path, index=False)
print(f"\n{'='*70}")
print(f" REPORT SAVED: {os.path.abspath(csv_path)}")
print(f"{'='*70}")

# Save GBR time series to CSV for reference / plotting
if df_gbr_ts is not None:
    ts_csv_path = os.path.join(OUTPUT_DIR, 'GBR_Monthly_TimeSeries_All5Vars.csv')
    df_gbr_ts.to_csv(ts_csv_path, index=False)
    print(f" GBR TIME SERIES SAVED: {os.path.abspath(ts_csv_path)}")
    print(f"{'='*70}")

print("\n[COMPLETE] BlueEco full data validation finished.")
print(df_summary[['Variable','overall_verdict','recommended_action']].to_string(index=False))